Dada a base CIFAR10 que contém imagens organizadas em 10 classes, considere a criação de um classificador.

Estratégia a serem avaliadas:

2) uso da rede pré-treinada como extrator de características e um modelo raso como classificador.

Questão em aberto: 
a) Trocar a rede CNN utilizada por uma mais simples, como a Mobilenet, impacta significativamente o resultado?

In [1]:
import tensorflow as tf
from tensorflow.keras import datasets
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input as preprocess_input_resnet50
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input as preprocess_input_mobilenetv2

I0000 00:00:1782680595.916816  105119 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# 1. Carregando CIFAR-10
print("Carregando CIFAR-10...")
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Transformando os rótulos de matriz (N, 1) para vetor (N,) exigido pelo scikit-learn
train_labels = train_labels.ravel()
test_labels = test_labels.ravel()

Carregando CIFAR-10...


In [ ]:
# 2. Pré-processamento
# Redes pré-treinadas exigem que os dados passem pela mesma normalização usada no ImageNet.
print("Pré-processando imagens para a ResNet50...")
train_images_preprocessed = preprocess_input_resnet50(train_images.astype(np.float32))
test_images_preprocessed = preprocess_input_resnet50(test_images.astype(np.float32))

# 3. Carregando a Rede Pré-treinada (Extratora)
# include_top=False: Remove a última camada densa original (que classificava 1000 classes do ImageNet)
# pooling='avg': Transforma as matrizes 2D em um vetor linear (Feature Vector)
print("Carregando ResNet50...")
extrator_features = ResNet50(weights='imagenet', 
                             include_top=False, 
                             pooling='avg', 
                             input_shape=(32, 32, 3))

# 4. Extração de Características
# Passamos as imagens pela rede. Ela NÃO vai treinar, apenas calcular os atributos profundos.
print("Extraindo características do Treinamento... (Pode levar 1-2 minutos)")
X_train_features = extrator_features.predict(train_images_preprocessed, batch_size=64)

print("Extraindo características do Teste...")
X_test_features = extrator_features.predict(test_images_preprocessed, batch_size=64)

print(f"Cada imagem agora é representada por um vetor de {X_train_features.shape[1]} atributos.")

Pré-processando imagens para a ResNet50...
Carregando ResNet50...


I0000 00:00:1782680524.690281   96028 gpu_device.cc:2042] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4055 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2060, pci bus id: 0000:01:00.0, compute capability: 7.5
E0000 00:00:1782680524.884544   96028 cuda_executor.cc:1168] [0] Failed to allocate device memory of 3.96GiB (4251975680 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:45

W0000 00:00:1782680531.748913   96028 cpu_allocator_impl.cc:82] Allocation of 9437184 exceeds 10% of free system memory.
W0000 00:00:1782680531.775078   96028 cpu_allocator_impl.cc:82] Allocation of 8388608 exceeds 10% of free system memory.
W0000 00:00:1782680531.814656   96028 cpu_allocator_impl.cc:82] Allocation of 9437184 exceeds 10% of free system memory.
W0000 00:00:1782680531.852949   96028 cpu_allocator_impl.cc:82] Allocation of 9437184 exceeds 10% of free system memory.


Extraindo características do Treinamento... (Pode levar 1-2 minutos)


In [11]:
# 5. Classificador Raso
# Usando um Perceptron de Múltiplas Camadas (MLP) com uma camada oculta de 500 neurônios
print("Treinando o MLP com as características extraídas... (Acompanhe a convergência)")
clf = MLPClassifier(hidden_layer_sizes=(500,), 
                    activation='relu', 
                    max_iter=150, 
                    random_state=42, 
                    verbose=True) # verbose=True permite ver a loss caindo

clf.fit(X_train_features, train_labels)

# 6. Avaliação
print("Avaliando o modelo no conjunto de teste...")
y_pred = clf.predict(X_test_features)
acc = accuracy_score(test_labels, y_pred)

print("-" * 30)
print(f"Acurácia final (ResNet50 + MLP): {acc*100:.2f}%")
print("-" * 30)

Treinando o MLP com as características extraídas... (Acompanhe a convergência)
Iteration 1, loss = 1.23826623
Iteration 2, loss = 0.82800937
Iteration 3, loss = 0.67621219
Iteration 4, loss = 0.53983402
Iteration 5, loss = 0.41677886
Iteration 6, loss = 0.30900708
Iteration 7, loss = 0.21583129
Iteration 8, loss = 0.15033126
Iteration 9, loss = 0.11097063
Iteration 10, loss = 0.09666676
Iteration 11, loss = 0.08197602
Iteration 12, loss = 0.08420084
Iteration 13, loss = 0.10132810
Iteration 14, loss = 0.09819482
Iteration 15, loss = 0.08846048
Iteration 16, loss = 0.06728068
Iteration 17, loss = 0.04464469
Iteration 18, loss = 0.03715718
Iteration 19, loss = 0.04776530
Iteration 20, loss = 0.06830148
Iteration 21, loss = 0.07023383
Iteration 22, loss = 0.07868495
Iteration 23, loss = 0.05579542
Iteration 24, loss = 0.04369759
Iteration 25, loss = 0.04403833
Iteration 26, loss = 0.04102833
Iteration 27, loss = 0.03693530
Iteration 28, loss = 0.05636388
Iteration 29, loss = 0.05838715
It

In [13]:
# 2. Pré-processamento
# Redes pré-treinadas exigem que os dados passem pela mesma normalização usada no ImageNet.
print("Pré-processando imagens para a MobileNetv2...")
train_images_preprocessed = preprocess_input_mobilenetv2(train_images.astype(np.float32))
test_images_preprocessed = preprocess_input_mobilenetv2(test_images.astype(np.float32))


print("Carregando MobileNetV2...")

extrator_features = MobileNetV2(weights='imagenet', 
                                include_top=False, 
                                pooling='avg', 
                                input_shape=(32, 32, 3))

# 4. Extração de Características
print("Extraindo características do Treinamento...")
X_train_features = extrator_features.predict(train_images_preprocessed, batch_size=64)

print("Extraindo características do Teste...")
X_test_features = extrator_features.predict(test_images_preprocessed, batch_size=64)

# O vetor da MobileNetV2 costuma ter 1280 características (menor que os 2048 da ResNet)
print(f"Cada imagem agora é representada por um vetor de {X_train_features.shape[1]} atributos.")

Carregando MobileNetV2...


/tmp/ipykernel_8022/3629688330.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  extrator_features = MobileNetV2(weights='imagenet',


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Extraindo características do Treinamento...
782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step
Extraindo características do Teste...
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
Cada imagem agora é representada por um vetor de 1280 atributos.


In [14]:
# 5. Classificador Raso
print("Treinando o MLP com as características extraídas...")
clf = MLPClassifier(hidden_layer_sizes=(500,), 
                    activation='relu', 
                    max_iter=150, 
                    random_state=42, 
                    verbose=True)

clf.fit(X_train_features, train_labels)

# 6. Avaliação
print("Avaliando o modelo no conjunto de teste...")
y_pred = clf.predict(X_test_features)
acc = accuracy_score(test_labels, y_pred)

print("-" * 30)
print(f"Acurácia final (MobileNetV2 + MLP): {acc*100:.2f}%")
print("-" * 30)

Treinando o MLP com as características extraídas...
Iteration 1, loss = 2.14617085
Iteration 2, loss = 2.07314110
Iteration 3, loss = 2.04727983
Iteration 4, loss = 2.02664590
Iteration 5, loss = 2.00909710
Iteration 6, loss = 1.99151838
Iteration 7, loss = 1.97529221
Iteration 8, loss = 1.95895278
Iteration 9, loss = 1.94471550
Iteration 10, loss = 1.92953777
Iteration 11, loss = 1.91460967
Iteration 12, loss = 1.90117776
Iteration 13, loss = 1.88780487
Iteration 14, loss = 1.87467110
Iteration 15, loss = 1.86206925
Iteration 16, loss = 1.84857380
Iteration 17, loss = 1.83664107
Iteration 18, loss = 1.82546031
Iteration 19, loss = 1.81413174
Iteration 20, loss = 1.80340874
Iteration 21, loss = 1.79349768
Iteration 22, loss = 1.78463662
Iteration 23, loss = 1.77294469
Iteration 24, loss = 1.76354611
Iteration 25, loss = 1.75509548
Iteration 26, loss = 1.74520779
Iteration 27, loss = 1.73791718
Iteration 28, loss = 1.72877240
Iteration 29, loss = 1.71967828
Iteration 30, loss = 1.711835

/home/jasimioni/.venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(


------------------------------
Acurácia final (MobileNetV2 + MLP): 21.85%
------------------------------
